# optimizer-init-params-list composite — cx3: optimizer __init__ + zero_grad(set_to_none=True)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `optimizer-init-params-list`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "optimizer-init-params-list"
DD_ATOM_IDS = ["optimizer-init-params-list", "zero-grad-set-none"]
DD_SUBTOPICS = ["PyTorch: Optimizer init", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Every training step in PyTorch looks like:
```
loss.backward()    # accumulates grads INTO p.grad (with += !).
optimizer.step()   # update params.
optimizer.zero_grad()  # clear grads for the next iteration.
```

The clear is mandatory. PyTorch's `backward()` ACCUMULATES into `p.grad` — if you skip the clear, the next batch's gradient is added on top of the previous one, and you train on a running sum of gradients (silent bug — loss looks like it's going down 'differently').

There are two ways to clear:
- `p.grad.zero_()` — in-place zero. Keeps the gradient tensor allocated.
- `p.grad = None` — drop the reference entirely. The NEXT backward call will re-allocate.

**zero-grad-set-none** is the second approach. PyTorch ≥ 1.7 made `set_to_none=True` the default in `Optimizer.zero_grad()` for two reasons: (1) saves memory between optim steps; (2) makes 'I forgot to zero_grad' show up as a `NoneType` crash instead of a silent accumulation bug.

**Composition with optimizer-init-params-list.** `zero_grad` iterates `self.params` — which only exists because `__init__` materialized the generator into a list. With a raw generator, the FIRST `zero_grad` call would empty the iterator and `.step()` would silently no-op.

**Anatomy.**
```python
class SGD:
    def __init__(self, params, lr):
        self.params = list(params)              # atom A.
        self.lr = lr

    def zero_grad(self, set_to_none=True):
        for p in self.params:
            if set_to_none:
                p.grad = None                   # atom B: drop the reference.
            else:
                if p.grad is not None:
                    p.grad.zero_()
```

### Composite Exercise — optimizer __init__ + zero_grad(set_to_none=True)

**Atoms exercised together**: `optimizer-init-params-list`, `zero-grad-set-none`

Implement `cx3_make_sgd_with_zero_grad()` — return a class `MySGD2`:

- `MySGD2(params, lr)`:
  - `self.params = list(params)` (atom A).
  - `self.lr = lr`.
- `MySGD2.zero_grad(self, set_to_none=True)`:
  - If `set_to_none`: for each param, set `p.grad = None` (atom B).
  - Else: for each param with non-None grad, call `p.grad.zero_()` in place.

(You do NOT need to implement `.step()` for this drill.)

The test checks: (a) after `zero_grad(set_to_none=True)`, every param's `.grad` is literally `None`; (b) after `zero_grad(set_to_none=False)`, every param's `.grad` is a tensor of zeros (not None); (c) the optimizer works when constructed from a generator (materialization atom); (d) calling `zero_grad` doesn't crash when some params already have `grad=None` (idempotent).

In [ ]:
def cx3_make_sgd_with_zero_grad():
    class MySGD2:
        def __init__(self, params, lr):
            # Atom A: materialize generator into a list — without this, zero_grad
            # would consume the iterator on the first call and step() would do nothing.
            self.params = list(params)
            self.lr = lr

        def zero_grad(self, set_to_none=True):
            for p in self.params:
                if set_to_none:
                    # Atom B: drop the reference. Next backward() will re-allocate.
                    p.grad = None
                else:
                    if p.grad is not None:
                        p.grad.zero_()  # in-place zero of the existing tensor.

    return MySGD2


<details><summary>Show solution — cx3</summary>

```python
def cx3_make_sgd_with_zero_grad():
    class MySGD2:
        def __init__(self, params, lr):
            # Atom A: materialize generator into a list — without this, zero_grad
            # would consume the iterator on the first call and step() would do nothing.
            self.params = list(params)
            self.lr = lr

        def zero_grad(self, set_to_none=True):
            for p in self.params:
                if set_to_none:
                    # Atom B: drop the reference. Next backward() will re-allocate.
                    p.grad = None
                else:
                    if p.grad is not None:
                        p.grad.zero_()  # in-place zero of the existing tensor.

    return MySGD2
```

Why `set_to_none=True` became the default in PyTorch 1.7+: (1) memory — the gradient tensors aren't kept alive between optim steps; (2) bug surfacing — if you forget to call `zero_grad` before the next `backward()`, the accumulation either happens (bad) or now crashes (good, because `p.grad` is None and the `+=` inside backward needs to allocate anyway). The `if p.grad is not None` guard in the `set_to_none=False` branch matters: calling `.zero_()` on `None` would crash.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx3',
        'subtopics': ["PyTorch: Optimizer init", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()